# FHIR Encounter Bronze-to-Silver Transformation

## Purpose

Transform raw FHIR Encounter resources from the Bronze layer into a
structured Silver Delta table.

### Source
`health_insurance.bronze.fhir_encounter_raw`

### Target
`health_insurance.silver.fhir_encounter`

### Responsibilities

- Parse raw FHIR Encounter JSON
- Extract encounter identifiers and status
- Extract encounter class and type
- Parse Patient, Practitioner, and Organization references
- Convert encounter start/end timestamps
- Derive encounter duration
- Preserve source lineage metadata

Formal data-quality enforcement is implemented separately in the
dedicated data-quality stage.



In [0]:

# stage configuration


CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_encounter_raw"
TARGET_TABLE = f"{CATALOG}.silver.fhir_encounter"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)

In [0]:

# Loading Bronze FHIR Encounter data


encounter_bronze_df = spark.table(SOURCE_TABLE)

print(f"Rows: {encounter_bronze_df.count():,}")
print(f"Columns: {len(encounter_bronze_df.columns)}")

encounter_bronze_df.printSchema()

display(encounter_bronze_df.limit(5))

In [0]:

# Infering combined FHIR Encounter JSON schema


schema_result = spark.sql(f"""
    SELECT schema_of_json_agg(raw_json) AS encounter_schema
    FROM {SOURCE_TABLE}
""").first()

encounter_schema = schema_result["encounter_schema"]

print("FHIR Encounter schema:")
print(encounter_schema)

In [0]:

# Parsing raw FHIR Encounter JSON


from pyspark.sql import functions as F

encounter_parsed_df = (
    encounter_bronze_df
    .withColumn(
        "encounter",
        F.from_json(
            F.col("raw_json"),
            encounter_schema
        )
    )
)

encounter_parsed_df.select("encounter.*").printSchema()

In [0]:

# Extracting core Encounter attributes


encounter_core_df = (
    encounter_parsed_df
    .select(
        F.col("encounter.id").alias("encounter_id"),
        F.col("encounter.status").alias("status"),

        F.col("encounter.class").alias("class_struct"),
        F.col("encounter.type").alias("type_array"),

        F.col("encounter.subject.reference").alias("patient_reference"),
        F.col("encounter.participant").alias("participant_array"),

        F.col("encounter.period.start").alias("start_datetime_raw"),
        F.col("encounter.period.end").alias("end_datetime_raw"),

        F.col("encounter.serviceProvider.reference").alias(
            "organization_reference"
        ),

        "_ingested_at",
        "_source_system",
        "_resource_type"
    )
)

In [0]:

# Extracting Encounter class


encounter_class_df = (
    encounter_core_df

    .withColumn(
        "encounter_class",
        F.col("class_struct.code")
    )

    .withColumn(
        "encounter_class_system",
        F.col("class_struct.system")
    )
)

In [0]:

# Extracting primary Encounter type


encounter_type_df = (
    encounter_class_df

    .withColumn(
        "primary_type",
        F.element_at(
            F.col("type_array"),
            1
        )
    )

    .withColumn(
        "encounter_type_code",
        F.element_at(
            F.col("primary_type.coding.code"),
            1
        )
    )

    .withColumn(
        "encounter_type",
        F.element_at(
            F.col("primary_type.coding.display"),
            1
        )
    )

    .withColumn(
        "encounter_type_system",
        F.element_at(
            F.col("primary_type.coding.system"),
            1
        )
    )
)

In [0]:

# Extracting primary Practitioner reference


encounter_participant_df = (
    encounter_type_df

    .withColumn(
        "primary_participant",
        F.element_at(
            F.col("participant_array"),
            1
        )
    )

    .withColumn(
        "practitioner_reference",
        F.col(
            "primary_participant.individual.reference"
        )
    )
)

In [0]:

# Parsing FHIR references


encounter_refs_df = (
    encounter_participant_df

    .withColumn(
        "patient_id",
        F.regexp_extract(
            F.col("patient_reference"),
            r"Patient/(.+)",
            1
        )
    )

    .withColumn(
        "practitioner_id",
        F.regexp_extract(
            F.col("practitioner_reference"),
            r"Practitioner/(.+)",
            1
        )
    )

    .withColumn(
        "organization_id",
        F.regexp_extract(
            F.col("organization_reference"),
            r"Organization/(.+)",
            1
        )
    )
)

In [0]:

# Converting Encounter timestamps


encounter_typed_df = (
    encounter_refs_df

    .withColumn(
        "start_datetime",
        F.to_timestamp("start_datetime_raw")
    )

    .withColumn(
        "end_datetime",
        F.to_timestamp("end_datetime_raw")
    )
)

In [0]:

# Deriving Encounter duration in minutes


encounter_enriched_df = (
    encounter_typed_df

    .withColumn(
        "encounter_duration_minutes",
        (
            F.col("end_datetime").cast("long")
            - F.col("start_datetime").cast("long")
        ) / 60
    )
)

encounter_enriched_df = (
    encounter_enriched_df
    .withColumn(
        "encounter_duration_minutes",
        F.col("encounter_duration_minutes").cast("int")
    )
)

In [0]:

# Standardizing Encounter attributes


encounter_standardized_df = (
    encounter_enriched_df

    .withColumn(
        "status",
        F.upper(F.trim("status"))
    )

    .withColumn(
        "encounter_class",
        F.upper(F.trim("encounter_class"))
    )
)

In [0]:

# Building final Silver Encounter dataset


encounter_silver_df = (
    encounter_standardized_df

    .select(
        "encounter_id",
        "patient_id",
        "practitioner_id",
        "organization_id",

        "status",

        "encounter_class",
        "encounter_class_system",

        "encounter_type_code",
        "encounter_type",
        "encounter_type_system",

        "start_datetime",
        "end_datetime",
        "encounter_duration_minutes",

        "_source_system",
        "_resource_type",
        "_ingested_at"
    )

    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp()
    )
)

In [0]:

# Inspecting Silver Encounter result


encounter_silver_df.printSchema()

display(
    encounter_silver_df.limit(20)
)

In [0]:

# Reconciling Bronze and Silver row counts


bronze_count = encounter_bronze_df.count()
silver_count = encounter_silver_df.count()

print(f"Bronze Encounters: {bronze_count:,}")
print(f"Silver Encounters: {silver_count:,}")
print(f"Difference: {bronze_count - silver_count:,}")

In [0]:

# Persisting FHIR Encounter Silver table


(
    encounter_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Silver table: {TARGET_TABLE}")

In [0]:
%sql
--quering the silver fhir_encounetr table


SELECT COUNT(*) AS encounter_count
FROM health_insurance.silver.fhir_encounter;

## Transformation Result

The raw FHIR Encounter resources were successfully transformed from
the Bronze layer into a structured Silver Delta table.

### Source

`health_insurance.bronze.fhir_encounter_raw`

### Target

`health_insurance.silver.fhir_encounter`

### Transformations Applied

- Parsed raw FHIR JSON into structured Spark data.
- Extracted encounter identifiers and status.
- Extracted FHIR encounter class and type coding.
- Parsed Patient, Practitioner, and Organization references.
- Converted encounter period fields to Spark timestamps.
- Derived encounter duration in minutes.
- Standardized selected categorical attributes.
- Preserved ingestion metadata and added Silver transformation metadata.

### Data Quality Boundary

This notebook performs structural transformation and standardization
only.

Formal validation of rules such as missing patient references,
invalid encounter durations, unsupported statuses, and broken
relationships is handled in the dedicated data-quality stage.

### Architecture

FHIR Encounter API  
↓  
`health_insurance.bronze.fhir_encounter_raw`  
↓  
**Bronze-to-Silver transformation**  
↓  
`health_insurance.silver.fhir_encounter`  
↓  
Data Quality  
↓  
Gold analytical model


### Status

**FHIR Encounter Bronze-to-Silver transformation: COMPLETE**